In [21]:
from google.colab import userdata

In [22]:
!pip install -qU langchain-google-genai

In [23]:
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [24]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [25]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

In [26]:
print(embeddings)

client=<google.genai.client.Client object at 0x7ffaf9a891d0> model='models/gemini-embedding-2-preview' task_type=None google_api_key=SecretStr('**********') credentials=None vertexai=None project=None location=None base_url=None additional_headers=None client_args=None api_version=None request_options=None output_dimensionality=None


In [27]:
text = "This is a tutorial on genai embeddings"

In [28]:
query_result = embeddings.embed_query(text)

In [29]:
print(query_result)

[-0.03748308, 0.024037272, 0.00045342924, 0.015140829, 0.00037873138, -0.006715732, -0.013426197, 0.011935201, 0.0077624675, -0.033183876, -0.010984841, 0.00057989685, -0.0008750525, -0.008827547, 0.01258898, -0.010557855, 0.05007269, -0.012148048, -0.0045624604, 0.007813201, -0.0047064763, -0.0066554714, -0.0009202256, -0.016814278, -0.017376544, 0.008258986, -0.0056683924, -0.014230753, 0.0065311533, 0.1435137, -0.013707335, -0.033695642, -0.019627454, -0.03113241, 0.0005291675, -0.0044112885, 0.014220028, -0.007389221, -0.00850649, -0.0002495004, 0.022520043, 0.00043751026, 0.016007034, -0.0063995197, -0.013805842, 0.0018741144, -0.013459077, 0.015969748, -0.006731193, -0.0106768515, -0.015969431, 0.014121427, -0.009068836, -0.030210778, 0.021369977, -0.0077965166, 0.008699815, -0.0067466064, 0.028908473, 0.008197792, -0.006538322, -0.028587162, 0.0021477472, 0.0067386576, 0.011446525, -0.0068783574, 0.0071097235, -0.015187259, 0.00928247, -0.00548818, 0.0018230723, -0.00057480275, 

In [30]:
len(query_result)

3072

In [31]:
import warnings

warnings.filterwarnings("ignore")

In [32]:
!pip install -U langchain-community

In [33]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('Speech.txt.txt', encoding='utf-8')
docs = loader.load()

In [34]:
print(docs)

[Document(metadata={'source': 'Speech.txt.txt'}, page_content='Good morning everyone,\n\nThink for a moment about how quickly our world changes. Just a couple of decades ago, accessing computing power required sitting at a bulky desktop or waiting on slow dial-up connections. Today, each of us carries a supercomputer in our pocket, connected to a global network of information and intelligence.\n\nWe are currently witnessing one of the most exciting shifts in human history: the rise of intelligent systems. We have moved beyond writing rigid, rule-based code to building systems that can understand context, parse natural language, and assist us in solving problems faster than ever before.\n\nWhat makes this era remarkable isn’t just the complexity of the technology, but its accessibility. Tools that once required massive supercomputing clusters and specialized research labs are now accessible right in our browsers with just a few lines of code. Whether it is building intelligent workflows

In [35]:
!pip install -U langchain-text-splitters

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Maximum number of characters per chunk
    chunk_overlap=50,      # Overlapping characters between chunks to preserve context
    separators=["\n\n", "\n", " ", ""]
)

# 2. Split your loaded documents
chunks = text_splitter.split_documents(docs)

In [37]:
print(chunks[0].page_content)

Good morning everyone,

Think for a moment about how quickly our world changes. Just a couple of decades ago, accessing computing power required sitting at a bulky desktop or waiting on slow dial-up connections. Today, each of us carries a supercomputer in our pocket, connected to a global network of information and intelligence.


In [39]:
chunk_vectors = []

for chunk in chunks:
    # Use .page_content to get the actual string
    vector = embeddings.embed_query(chunk.page_content)
    chunk_vectors.append(vector)

print("Done! Total vectors:", len(chunk_vectors))

Done! Total vectors: 5


In [40]:
!pip install -U langchain-chroma

In [41]:
from langchain_chroma import Chroma

In [43]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="speech_database"
)
print("Chunks successfully stored in Chroma DB!")

Chunks successfully stored in Chroma DB!


In [59]:
query = "carrot"

In [60]:
matched_docs = vector_store.similarity_search(query, k=2)

In [61]:
for i, doc in enumerate(matched_docs, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
The future isn’t something we wait for; it’s something we build one line of code and one idea at a time.

Thank you.

--- Result 2 ---
The future isn’t something we wait for; it’s something we build one line of code and one idea at a time.

Thank you.
